# 03. 2단계 예측 모델 — 검증, 과거 평균 비교, 최종 모델 저장

baseline(01)이 달력 효과를 처리하고, 이 모델은 그 위에서 **기상이 만든 변화 y**를 예측합니다. 예보가 나오면 날짜별 y를 예측하고, 실제와 같은 판정·3축 공식을 적용해 취약 순위를 미리 만듭니다(모델이 3축을 직접 예측하지 않음).

| 항목 | 설정 |
|---|---|
| 모델 | LightGBM 회귀, Huber 손실(α=0.1). L2·L1·Huber 비교 후 특보일 오차 개선율 + 업종 순위상관으로 선택 |
| 하이퍼파라미터 | 트리 400, 학습률 0.03, 잎 31, 잎당 최소 200행, 행·열 샘플링 0.8, L2 규제 1.0 |
| 변수 14개 | 지역·업종·시간대 / 시간대 최고기온·강수·강수시간·습도·풍속 / 일 최고기온·일 강수 / 전3일강수·폭염연속일수 / 폭염·호우 등급 |
| 학습 데이터 | 평시 + 특보일 전체, 휴무 칸·특수일(연휴 블록 ±1일) 제외 |
| 검증 | 이벤트 묶음을 하나씩 통째로 빼고 학습(검증 날짜 ±3일 학습 제외) |

이 노트북이 하는 일: ① 이벤트 제외 예측 ② 과거 평균·평소대로 예측과 비교(사전 판단 규칙) ③ 예측 순위표 ④ 전체 데이터로 최종 모델 학습·저장

---

## 과거 평균 예측과 비교 — 2단계 모델을 쓰는 이유

**질문**: 예보가 나왔을 때 "어떤 업종이 얼마나 줄지"를 맞히는 데, 기상 모델(팀원 LightGBM Huber)이 **업종별 과거 평균 반응**만 쓰는 것보다 나은가?

**비교하는 예측 세 가지** (모두 같은 검증: 이벤트 묶음을 하나씩 통째로 빼고 학습, 검증 날짜 ±3일 학습 제외)

| 예측 | 내용 |
|---|---|
| 모델 | 팀원 LightGBM Huber(α=0.1), 변수 14개(지역·업종·시간대 + 기상 9개 + 특보 등급 2개). 기온·강수의 **크기**와 이어진 날수를 반영 |
| 과거 평균 | 학습 기간에서 같은 업종·시간대의 **같은 종류 특보일(폭염일/호우일) 평균 y**. 특보 없는 날은 특보 없는 날의 평균 y. 표본이 없으면 업종 평균 → 0 |
| 평소대로 | y = 0 (특보가 와도 평소와 같다고 보는 예측) — 둘 다 이것보다는 나아야 함 |

**평가 지표** (이벤트 분석일 = 특보 기간에서 특수일 뺀 날)
1. **업종 순위 맞히기**(주 지표): 이벤트마다 업종별 기간 차이율 $\sum(S-B)/\sum B$의 실제·예측 순위상관 → 이벤트 평균, 모델이 이긴 이벤트 수
2. 칸 단위 오차: 특보일 시간대 y의 MAE
3. 축별 순위상관: 이벤트×업종 Impact·Cumulative
4. 취약 후보 찾기: 예측 차이율을 **실제와 같은 평시 범위**로 판정 → 반복 조건 → 실제 후보(유형나누기 결과) 재현율·정밀도
5. 상위 10 겹침: 실제 3축 순위 상위 10 vs 예측 2축(Impact·Cumulative, 판정 없이) 상위 10

**사전 판단 규칙 (실행 전에 정함 — 결과를 보고 바꾸지 않음)**
- 지역·기상별로 **주 지표의 이벤트 평균이 과거 평균보다 높고, 이긴 이벤트가 절반 이상**이면 "모델이 과거 평균보다 업종 순위를 더 잘 맞힌다"고 표현한다.
- 둘 중 하나만 만족하면 "비슷하다", 둘 다 아니면 "과거 평균이 낫거나 같다"로 표현한다.
- 이벤트가 적어(폭염 4개, 호우 5~6개) 통계적 유의성 검정은 하지 않고, 이벤트별 결과를 모두 보여 준다.

**입력**: `38_최종baseline_V4.csv`, 기상 병합 CSV(팀원 모델과 같은 파일), `유형나누기_결과.xlsx`(실제 판정·평시 범위·순위를 그대로 사용 → **유형나누기를 먼저 실행**)

In [ ]:
# 0. 설정
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/big_contest')
try:
    from google.colab import drive; drive.mount('/content/drive')
    import subprocess, sys; subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib', 'lightgbm'])
except ImportError:
    pass
import re, time, warnings
import numpy as np, pandas as pd, lightgbm as lgb, matplotlib.pyplot as plt
from scipy.stats import spearmanr
try: import koreanize_matplotlib
except ImportError: print('※ 한글 글꼴 패키지가 없어 그래프 글자가 깨질 수 있습니다')
warnings.filterwarnings('ignore')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

def find(*patterns):
    for pat in patterns:
        hits = sorted(ROOT.rglob(pat), key=lambda p: p.stat().st_mtime)
        if hits: print('읽는 파일:', hits[-1]); return hits[-1]
    raise FileNotFoundError(f'{patterns} 을(를) {ROOT} 아래에서 찾지 못했습니다')
V4_PATH = find('38_최종baseline_V4.csv')
WEATHER_PATH = find('강남_춘천_기상데이터_병합(6).csv', '*기상*병합*.csv', '*기상*정제*.csv')   # 팀원 모델과 같은 파일
TYPE_PATH = find('유형나누기_결과.xlsx')
OUT = ROOT/'results'/'2단계모델'; OUT.mkdir(parents=True, exist_ok=True)

SPECIAL2_DAYS = pd.DatetimeIndex(np.concatenate([pd.date_range(a, b) for a, b in
                 [('2025-08-14', '2025-08-18'), ('2025-10-02', '2025-10-10'), ('2025-12-24', '2025-12-26')]]))
SCOPE_CHECK = ['보험', '통신요금(이동,시내전화)', '결제대행(PG)']
GAP_MERGE = 0                     # 나눈 버전 (유형나누기와 같음)
REPEAT_MIN, REPEAT_SHARE = 2, 0.5
BUFFER, SEED, TOPK = 3, 42, 10
LGB_PARAMS = dict(n_estimators=400, learning_rate=0.03, num_leaves=31, min_child_samples=200,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                  cat_smooth=50, random_state=SEED, verbose=-1, n_jobs=-1,
                  objective='huber', alpha=0.1)      # 팀원 노트북 설정 그대로
PRED_NAMES = ['모델', '과거 평균', '평소대로']

COL = dict(cat=['#2a78d6', '#eb6834', '#1baf7a', '#eda100'], ink='#0b0b0b', ink2='#52514e', grid='#e4e3df')
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': COL['ink2'],
                     'axes.labelcolor': COL['ink2'], 'xtick.color': COL['ink2'], 'ytick.color': COL['ink2'],
                     'axes.grid': True, 'grid.color': COL['grid'], 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'axes.unicode_minus': False, 'figure.dpi': 110})

In [ ]:
# 1. 데이터 준비 (팀원 노트북·3축 재점검과 같은 처리)
P = pd.read_csv(V4_PATH, encoding='utf-8-sig', parse_dates=['날짜'])
P['특수일2'] = P['날짜'].isin(SPECIAL2_DAYS)
ALL_DATES = pd.date_range(P['날짜'].min(), P['날짜'].max())
dd = P.drop_duplicates(['대상', '날짜'])
D = (P[~P['휴무']].groupby(['대상', '업종', 'gkey', '날짜'])
     .agg(S=('S', 'sum'), B=('B', 'sum')).reset_index())

W = pd.read_csv(WEATHER_PATH, encoding='utf-8-sig', parse_dates=['일시'])
W['날짜'] = W['일시'].dt.normalize(); W['hour'] = W['일시'].dt.hour
SLOTS = sorted(P['slot'].unique(), key=lambda s: int(re.findall(r'\d+', s)[0]))
starts = [int(re.findall(r'\d+', s)[0]) for s in SLOTS]
W['slot'] = W['hour'].map(lambda h: SLOTS[max(i for i, s in enumerate(starts) if s <= h)])
WS = W.groupby(['대상', '날짜', 'slot']).agg(시간대습도=('습도(%)', 'mean'), 시간대풍속=('풍속(m/s)', 'mean')).reset_index()
CAL = pd.MultiIndex.from_product([sorted(P['대상'].unique()), ALL_DATES], names=['대상', '날짜']).to_frame(index=False)
CAL = CAL.join(W.groupby(['대상', '날짜'])['강수량(mm)'].sum().rename('r'), on=['대상', '날짜'])
CAL['전3일강수'] = CAL.groupby('대상')['r'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())
heat = dd.set_index(['대상', '날짜'])['폭염등급']
CAL['h'] = CAL.set_index(['대상', '날짜']).index.map(heat).fillna(0).to_numpy() > 0
CAL['폭염연속일수'] = CAL.groupby('대상')['h'].transform(lambda s: s.groupby((~s).cumsum()).cumsum()).astype(int)
P = P.merge(WS, on=['대상', '날짜', 'slot'], how='left').merge(CAL[['대상', '날짜', '전3일강수', '폭염연속일수']], on=['대상', '날짜'], how='left')
FEATS = ['대상', '업종', 'slot', '시간대최고기온', '시간대강수', '시간대강수시간', '시간대습도', '시간대풍속',
         '일최고기온', '일강수', '전3일강수', '폭염연속일수', '폭염등급', '호우등급']
for c in ['대상', '업종', 'slot']: P[c] = P[c].astype('category')
A = P[(~P['휴무']) & (~P['특수일2']) & P['y'].notna()].reset_index(drop=True)
A['특보종류'] = np.where(A['폭염등급'] > 0, '폭염', np.where(A['호우등급'] > 0, '호우', '없음'))
print(f'2단계 대상 {len(A):,}행, 업종 {A["gkey"].nunique()}개')

# 실제 결과(유형나누기) — 판정·평시 범위·순위를 그대로 씀
EPI_ACT = pd.read_excel(TYPE_PATH, sheet_name='이벤트별')
S_ACT = pd.read_excel(TYPE_PATH, sheet_name='전체업종_판정')
print('실제 취약 후보:', S_ACT[S_ACT['후보']].groupby(['대상', '종류']).size().to_dict())

In [ ]:
# 2. 이벤트(나눈 버전)와 검증 묶음
def runs(s):
    out, cur = [], []
    for d, v in s.items():
        if v: cur.append(d)
        elif cur: out.append(cur); cur = []
    if cur: out.append(cur)
    return out
EP = []
for reg in sorted(dd['대상'].unique()):
    for typ, col in [('폭염', '폭염등급'), ('호우', '호우등급')]:
        s = dd[dd['대상'] == reg].set_index('날짜')[col].reindex(ALL_DATES).fillna(0) > 0; merged = []
        for r in runs(s):
            if merged and (r[0] - merged[-1][-1]).days - 1 <= GAP_MERGE: merged[-1] = list(pd.date_range(merged[-1][0], r[-1]))
            else: merged.append(list(r))
        for i, days in enumerate(merged, 1):
            EP.append(dict(대상=reg, 종류=typ, 에피소드=f'{typ}{i}', 시작=days[0], 종료=days[-1], 날짜=days,
                           분석일=[d for d in days if d not in SPECIAL2_DAYS]))
EP = pd.DataFrame(EP)
print('이벤트 수:', EP.groupby(['대상', '종류']).size().to_dict())

# 날짜가 겹치는 이벤트(두 지역 같은 폭염 등)는 한 묶음으로 같이 뺌
spans = sorted([(e['시작'], e['종료']) for _, e in EP.iterrows()]); clusters = []
for a, b in spans:
    if clusters and a <= clusters[-1][1] + pd.Timedelta(days=BUFFER): clusters[-1][1] = max(clusters[-1][1], b)
    else: clusters.append([a, b])
CL = pd.DataFrame(clusters, columns=['시작', '종료']); CL.index = CL.index + 1
print(f'검증 묶음 {len(CL)}개'); print(CL.assign(시작=CL['시작'].dt.strftime('%m/%d'), 종료=CL['종료'].dt.strftime('%m/%d')).T.to_string())

In [ ]:
# 3. 이벤트 제외 예측: 모델 / 과거 평균 / 평소대로
adates = A['날짜'].values.astype('datetime64[D]')
def near(dates):
    arr = np.array(sorted(dates), dtype='datetime64[D]'); m = np.zeros(len(A), bool)
    for o in range(-BUFFER, BUFFER + 1): m |= np.isin(adates, arr + np.timedelta64(o, 'D'))
    return m

def past_mean(trn, te):
    tr = A.loc[trn]
    m1 = tr.groupby(['gkey', 'slot', '특보종류'], observed=True)['y'].mean()
    m2 = tr.groupby(['gkey', '특보종류'], observed=True)['y'].mean()
    t = A.loc[te, ['gkey', 'slot', '특보종류']]
    v = m1.reindex(pd.MultiIndex.from_frame(t)).to_numpy()
    v2 = m2.reindex(pd.MultiIndex.from_frame(t[['gkey', '특보종류']])).to_numpy()
    return np.where(np.isfinite(v), v, np.where(np.isfinite(v2), v2, 0.0))

PRED = {k: np.full(len(A), np.nan) for k in PRED_NAMES}
t0 = time.time()
for i, c in CL.iterrows():
    tset = set(pd.date_range(c['시작'], c['종료']))
    te = A['날짜'].isin(tset).values
    if te.sum() == 0: continue
    trn = ~near(tset)
    m = lgb.LGBMRegressor(**LGB_PARAMS).fit(A.loc[trn, FEATS], A.loc[trn, 'y'])
    PRED['모델'][te] = m.predict(A.loc[te, FEATS])
    PRED['과거 평균'][te] = past_mean(trn, te)
    PRED['평소대로'][te] = 0.0
    print(f'  묶음 {i}: 검증 {te.sum():,}행, 학습 {trn.sum():,}행')
print(f'예측 완료 {time.time()-t0:.0f}초')

In [ ]:
# 4. 이벤트×업종 지표 (실제는 유형나누기 결과, 예측은 같은 공식)
BD = D.set_index(['gkey', '날짜'])['B']
DELTA = {k: A.assign(d=v * A['B_하루합']).groupby(['gkey', '날짜'], observed=True)['d'].sum() for k, v in PRED.items()}
ACT = EPI_ACT.set_index(['대상', '종류', '에피소드', 'gkey'])
rows = []
for _, e in EP.iterrows():
    full = pd.DatetimeIndex(e['날짜']); an = set(e['분석일'])
    sub = D[(D['대상'] == e['대상']) & D['날짜'].isin(an) & (D['B'] > 0)]
    for g, z in sub.groupby('gkey'):
        key = (e['대상'], e['종류'], e['에피소드'], g)
        if key not in ACT.index: continue
        a = ACT.loc[key]; z = z.sort_values('날짜'); win = min(3, len(full))
        idx = pd.MultiIndex.from_arrays([[g] * len(z), z['날짜']])
        row = dict(대상=e['대상'], 종류=e['종류'], 에피소드=e['에피소드'], gkey=g, 업종=g.split('|')[1],
                   실제_차이율=a['차이율'], 실제_Impact=a['Impact'], 실제_Cumulative=a['Cumulative'], 실제_판정=a['판정'],
                   평시하한=a['평시하한'], 평시상한=a['평시상한'])
        for k in PRED_NAMES:
            dl = DELTA[k].reindex(idx).to_numpy()
            if not np.isfinite(dl).all(): continue
            yy = pd.Series(dl / z['B'].to_numpy(), index=z['날짜']).reindex(full)
            rate = dl.sum() / z['B'].sum()
            row.update({f'{k}_차이율': rate, f'{k}_Impact': yy.rolling(win, min_periods=win).mean().min(),
                        f'{k}_Cumulative': dl.sum() / z['B'].mean(),
                        f'{k}_판정': ('감소' if rate < a['평시하한'] else '증가' if rate > a['평시상한'] else '판단 불가')
                                    if np.isfinite([a['평시하한'], a['평시상한']]).all() else '계산 불가'})
        rows.append(row)
EV = pd.DataFrame(rows)
EV['분석범위검토'] = EV['업종'].isin(SCOPE_CHECK)
print(f'이벤트×업종 {len(EV):,}행')

## 결과
주 지표는 **이벤트마다 업종 순위를 얼마나 맞히는가**입니다. "평소대로"는 모든 업종을 0으로 예측해서 순위가 없으므로 순위상관은 계산되지 않습니다(칸 오차 비교용).

In [ ]:
# 5. 주 지표 — 이벤트별 업종 순위상관 (분석범위 보류 업종 제외)
def rho(a, b):
    z = pd.DataFrame({'a': a, 'b': b}).dropna()
    return float(spearmanr(z.a, z.b).statistic) if len(z) >= 5 and z.a.nunique() > 1 and z.b.nunique() > 1 else np.nan
E = EV[~EV['분석범위검토']]
per_ev = []
for (reg, typ, ep), z in E.groupby(['대상', '종류', '에피소드'], sort=False):
    per_ev.append(dict(대상=reg, 종류=typ, 에피소드=ep, 업종수=len(z),
                       모델=rho(z['실제_차이율'], z['모델_차이율']), 과거평균=rho(z['실제_차이율'], z['과거 평균_차이율'])))
PE = pd.DataFrame(per_ev); PE['모델 승'] = PE['모델'] > PE['과거평균']
print('[이벤트별 업종 순위상관] 실제 차이율 vs 예측 차이율'); print(PE.round(3).to_string(index=False))

summ = []
for (reg, typ), z in PE.groupby(['대상', '종류']):
    m, p = z['모델'].mean(), z['과거평균'].mean(); win, n = int(z['모델 승'].sum()), int(z[['모델', '과거평균']].notna().all(axis=1).sum())
    c1, c2 = m > p, win >= n / 2
    verdict = '모델이 업종 순위를 더 잘 맞힘' if c1 and c2 else ('비슷함' if c1 or c2 else '과거 평균이 낫거나 같음')
    summ.append(dict(대상=reg, 종류=typ, 이벤트=n, 모델_평균=m, 과거평균_평균=p, 모델승=f'{win}/{n}', 판단=verdict))
SUMM = pd.DataFrame(summ)
print('\n[사전 판단 규칙 적용]'); print(SUMM.round(3).to_string(index=False))

In [ ]:
# 6. 보조 지표
# (a) 특보일 칸 단위 y MAE — 분석일, 분석범위 보류 제외
ev_days = {(r, t): set(sum(z['분석일'].tolist(), [])) for (r, t), z in EP.groupby(['대상', '종류'])}
mae = []
for (reg, typ), days in ev_days.items():
    msk = (A['대상'] == reg).values & A['날짜'].isin(days).values & ~A['업종'].isin(SCOPE_CHECK).values
    row = dict(대상=reg, 종류=typ, 칸수=int(msk.sum()))
    for k in PRED_NAMES: row[f'{k}_MAE'] = np.nanmean(np.abs(A.loc[msk, 'y'].to_numpy() - PRED[k][msk]))
    row['모델 개선율(평소대로 대비)'] = 1 - row['모델_MAE'] / row['평소대로_MAE']
    row['과거평균 개선율(평소대로 대비)'] = 1 - row['과거 평균_MAE'] / row['평소대로_MAE']
    mae.append(row)
MAE = pd.DataFrame(mae)
print('[칸 단위 y MAE] 작을수록 좋음'); print(MAE.round(4).to_string(index=False))

# (b) 이벤트×업종 축별 순위상관
ax = []
for (reg, typ), z in E.groupby(['대상', '종류']):
    for a_ in ['Impact', 'Cumulative']:
        ax.append(dict(대상=reg, 종류=typ, 축=a_, **{k: rho(z[f'실제_{a_}'], z[f'{k}_{a_}']) for k in ['모델', '과거 평균']}))
AXV = pd.DataFrame(ax); print('\n[이벤트×업종 축별 순위상관]'); print(AXV.round(3).to_string(index=False))

# (c) 취약 후보 찾기 + (d) 상위 10 겹침
act_c = S_ACT[S_ACT['후보']]
rec = []
for (reg, typ), z in E.groupby(['대상', '종류']):
    truth = set(act_c.query('대상==@reg and 종류==@typ')['gkey'])
    top_a = set(act_c.query('대상==@reg and 종류==@typ').sort_values(['순위', 'gkey']).head(TOPK)['gkey']); k_ = len(top_a)
    row = dict(대상=reg, 종류=typ, 실제후보=len(truth), 상위k=k_)
    for k in ['모델', '과거 평균']:
        g = z.groupby('gkey').agg(n=(f'{k}_판정', lambda s: (s != '계산 불가').sum()), dec=(f'{k}_판정', lambda s: (s == '감소').sum()),
                                  I=(f'{k}_Impact', 'min'), C=(f'{k}_Cumulative', 'sum'))
        pc = set(g[(g['dec'] >= REPEAT_MIN) & (g['dec'] / g['n'].clip(lower=1) >= REPEAT_SHARE)].index)
        sc = (g['I'].rank(ascending=False, pct=True) + g['C'].rank(ascending=False, pct=True)) / 2
        top_p = set(sc.sort_values(ascending=False).head(k_).index)
        row.update({f'{k}_예측후보': len(pc), f'{k}_재현율': len(truth & pc) / len(truth) if truth else np.nan,
                    f'{k}_정밀도': len(truth & pc) / len(pc) if pc else np.nan, f'{k}_상위겹침': len(top_a & top_p)})
    rec.append(row)
REC = pd.DataFrame(rec)
print('\n[취약 후보 찾기 · 상위 겹침] 재현율 = 실제 후보 중 예측도 후보로 찾은 비율, 정밀도 = 예측 후보 중 실제 후보 비율')
print(REC.round(3).to_string(index=False))

In [ ]:
# 7. 한눈에 보기 + 저장
VIEW = SUMM[['대상', '종류', '이벤트', '모델_평균', '과거평균_평균', '모델승', '판단']].merge(
    MAE[['대상', '종류', '모델_MAE', '과거 평균_MAE', '평소대로_MAE']], on=['대상', '종류']).merge(
    REC[['대상', '종류', '실제후보', '모델_재현율', '과거 평균_재현율', '모델_상위겹침', '과거 평균_상위겹침', '상위k']], on=['대상', '종류'])
VIEW = VIEW.rename(columns={'모델_평균': '순위상관_모델', '과거평균_평균': '순위상관_과거평균'})
print('[한눈에 보기]'); print(VIEW.round(3).to_string(index=False))

with pd.ExcelWriter(OUT/'2단계모델_검증결과.xlsx') as xw:
    VIEW.to_excel(xw, sheet_name='한눈에', index=False)
    PE.to_excel(xw, sheet_name='이벤트별_순위상관', index=False)
    MAE.to_excel(xw, sheet_name='칸MAE', index=False)
    AXV.to_excel(xw, sheet_name='축별_순위상관', index=False)
    REC.to_excel(xw, sheet_name='후보찾기_상위겹침', index=False)
    EV.to_excel(xw, sheet_name='이벤트x업종', index=False)
print('\n저장:', OUT/'2단계모델_검증결과.xlsx')

## 예측 순위표
이벤트를 빼고 학습한 모델의 예측으로 만든 **예측 2축(Impact·Cumulative) 순위**를 실제 3축 순위 옆에 놓습니다. 예측 Recovery는 모델이 특보 종료를 몰라 쓰지 않습니다. `예측후보` = 예측 차이율을 실제와 같은 평시 범위로 판정했을 때 반복 조건을 만족한 업종, `예측후보순위` = 예측 후보끼리의 2축 순위(실제 순위와 같은 방식), `예측순위` = 판정 없이 전체 업종을 2축으로 줄 세운 순위입니다. 판정 없는 순위는 평소 출렁임이 큰 업종(악기/음반 등)이 위로 올라오기 쉬워서, 화면에는 예측 후보만 보여 주고 엑셀에 전체를 저장합니다.

In [ ]:
# 8. 예측 순위표 (분석범위 보류 업종 제외)
act = S_ACT[['대상', '종류', 'gkey', '후보', '순위', '반복']].rename(columns={'순위': '실제순위', '반복': '실제반복', '후보': '실제후보'})
rank_rows = []
for (reg, typ), z in E.groupby(['대상', '종류']):
    g = z.groupby('gkey').agg(I=('모델_Impact', 'min'), Cu=('모델_Cumulative', 'sum'),
                              dec=('모델_판정', lambda s: (s == '감소').sum()), n=('모델_판정', lambda s: (s != '계산 불가').sum()))
    g['예측점수'] = (g['I'].rank(ascending=False, pct=True) + g['Cu'].rank(ascending=False, pct=True)) / 2
    g['예측순위'] = g['예측점수'].rank(ascending=False, method='min')
    g['예측후보'] = (g['dec'] >= REPEAT_MIN) & (g['dec'] / g['n'].clip(lower=1) >= REPEAT_SHARE)
    g['예측반복'] = g['dec'].astype(int).astype(str) + '/' + g['n'].astype(int).astype(str)
    c = g['예측후보']   # 예측 후보끼리 2축 순위 (실제 순위와 같은 방식: 후보 안에서 백분위 평균)
    g['예측후보순위'] = np.nan
    if c.any():
        sc = (g.loc[c, 'I'].rank(ascending=False, pct=True) + g.loc[c, 'Cu'].rank(ascending=False, pct=True)) / 2
        g.loc[c, '예측후보순위'] = sc.rank(ascending=False, method='min')
    rank_rows.append(g.reset_index().assign(대상=reg, 종류=typ))
PR = pd.concat(rank_rows).merge(act, on=['대상', '종류', 'gkey'], how='left')
PR['업종'] = PR['gkey'].str.split('|').str[1]
PR['예측Impact(%)'] = (PR['I'] * 100).round(1); PR['예측Cumulative(하루치)'] = PR['Cu'].round(2)
PR = PR.sort_values(['대상', '종류', '예측후보순위', '예측순위'])
SHOWP = ['예측후보순위', '예측순위', '업종', '예측Impact(%)', '예측Cumulative(하루치)', '예측반복', '예측후보', '실제순위', '실제반복', '실제후보']
for (reg, typ), t in PR.groupby(['대상', '종류']):
    tc = t[t['예측후보']]
    print(f'\n■ {reg} {typ} — 예측 후보 {len(tc)}개 (실제 후보 {int(t["실제후보"].fillna(False).astype(bool).sum())}개)')
    print('없음' if tc.empty else tc.head(TOPK)[SHOWP].to_string(index=False))
PR[['대상', '종류'] + SHOWP].to_excel(OUT/'2단계모델_예측순위.xlsx', index=False)
print('\n저장:', OUT/'2단계모델_예측순위.xlsx')

## 최종 모델 저장 (예보 적용용)
검증이 끝난 설정 그대로 **전체 데이터**로 한 번 더 학습해 저장합니다. 예보에 쓸 때는 같은 14개 변수(같은 이름·범주)를 만든 표를 넣어 y를 예측하고, `y × 그날 업종 하루 baseline 합(B_하루합)`을 시간대별로 더하면 예상 차액이 됩니다.

```python
m = lgb.Booster(model_file='2단계모델_LGBM_Huber.txt')
y_hat = m.predict(X_forecast[FEATS])   # 대상·업종·slot은 category 형식
```

In [ ]:
# 9. 전체 데이터로 최종 모델 학습·저장 + 변수 중요도
final = lgb.LGBMRegressor(**LGB_PARAMS).fit(A[FEATS], A['y'])
final.booster_.save_model(str(OUT/'2단계모델_LGBM_Huber.txt'))
imp = pd.Series(final.booster_.feature_importance('gain'), index=FEATS).sort_values(ascending=False)
print('[변수 중요도, gain 비중 %]'); print((imp / imp.sum() * 100).round(1).to_string())
print('\n저장:', OUT/'2단계모델_LGBM_Huber.txt')

In [ ]:
# 10. 그래프 — 이벤트별 업종 순위상관 (모델 vs 과거 평균)
groups = list(PE.groupby(['대상', '종류']))
fig, axes = plt.subplots(1, len(groups), figsize=(3.6 * len(groups), 3.8), squeeze=False, sharey=True)
for ax, ((reg, typ), z) in zip(axes[0], groups):
    x = np.arange(len(z))
    for j, (_, r) in enumerate(z.iterrows()):
        ax.plot([j, j], [r['과거평균'], r['모델']], color=COL['grid'], lw=2, zorder=1)
    ax.scatter(x, z['과거평균'], s=46, color=COL['cat'][1], edgecolor='white', linewidth=1.5, zorder=3, label='과거 평균')
    ax.scatter(x, z['모델'], s=46, color=COL['cat'][0], edgecolor='white', linewidth=1.5, zorder=3, label='모델')
    ax.axhline(0, color=COL['ink2'], lw=1)
    ax.set_xticks(x, z['에피소드'], fontsize=8)
    ax.set_title(f'{reg[:2]} {typ}', fontsize=10, color=COL['ink'], loc='left')
axes[0, 0].set_ylabel('업종 순위상관 (실제 vs 예측)')
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, fontsize=9, loc='upper right', ncol=2, bbox_to_anchor=(1, 1.08))
fig.suptitle('이벤트마다 업종 피해 순위를 얼마나 맞혔나 (이벤트를 빼고 학습한 예측)', fontsize=11, color=COL['ink'], x=0.01, ha='left', y=1.08)
fig.tight_layout(); fig.savefig(OUT/'그래프9_과거평균비교.png', bbox_inches='tight'); plt.show()
print('그래프 저장 폴더:', OUT)